In [1]:
#Install Dependencies
!pip install langchain langchain-community langchain-groq
!pip install faiss-cpu sentence-transformers
!pip install streamlit pyngrok groq
!pip install pypdf wikipedia beautifulsoup4 requests
!npm install -g localtunnel
print('✅ All packages installed!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-

In [2]:
# setting api key
import os
from getpass import getpass

GROQ_API_KEY = getpass('Enter Your Api Key')
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
print('✅ API key set!')

Enter Your Api Key··········
✅ API key set!


In [3]:
#Build the Knowledge Base
import wikipedia
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# -------------------------------------------------------
# 📝 CUSTOMIZE: Add/change topics for your knowledge base
# -------------------------------------------------------
TOPICS = [
    "Artificial intelligence",
    "Machine learning",
    "Natural language processing",
    "Deep learning",
    "Transformer (deep learning architecture)",
    "Large language model",
    "ChatGPT",
    "Retrieval-augmented generation"
]

print('📥 Fetching Wikipedia articles...')
raw_docs = []

for topic in TOPICS:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        doc = Document(
            page_content=page.content[:8000],  # First 8000 chars per article
            metadata={"source": "Wikipedia", "title": topic, "url": page.url}
        )
        raw_docs.append(doc)
        print(f'  ✅ {topic}')
    except Exception as e:
        print(f'  ⚠️  Skipped "{topic}": {e}')

# Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=['\n\n', '\n', '. ', ' ']
)
chunks = splitter.split_documents(raw_docs)

print(f'\n📊 Knowledge Base Summary:')
print(f'   • Articles loaded : {len(raw_docs)}')
print(f'   • Total chunks    : {len(chunks)}')
print(f'   • Avg chunk size  : {sum(len(c.page_content) for c in chunks)//len(chunks)} chars')

📥 Fetching Wikipedia articles...
  ✅ Artificial intelligence
  ✅ Machine learning
  ✅ Natural language processing
  ✅ Deep learning
  ✅ Transformer (deep learning architecture)
  ✅ Large language model
  ✅ ChatGPT
  ✅ Retrieval-augmented generation

📊 Knowledge Base Summary:
   • Articles loaded : 8
   • Total chunks    : 127
   • Avg chunk size  : 504 chars


In [4]:
#Create Vector Store (FAISS)
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print('🔄 Loading embedding model (this may take ~1 minute)...')

# Free, runs locally — no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print('🏗️  Building FAISS vector index...')
vectorstore = FAISS.from_documents(chunks, embeddings)

# Save the index so Streamlit can load it
vectorstore.save_local('faiss_index')

print('✅ Vector store created and saved!')

# Quick sanity test
test_results = vectorstore.similarity_search('What is a transformer model?', k=2)
print(f'\n🔍 Quick test — Top result preview:')
print(f'   {test_results[0].page_content[:200]}...')

🔄 Loading embedding model (this may take ~1 minute)...


/tmp/ipykernel_6201/52906539.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🏗️  Building FAISS vector index...
✅ Vector store created and saved!

🔍 Quick test — Top result preview:
   The original version of the transformer architecture was proposed in the 2017 paper "Attention Is All You Need" by researchers at Google. The predecessors of transformers were developed as an improvem...


In [4]:
!pip uninstall -y langchain langchain-core langchain-community
!pip install langchain==0.2.16 langchain-core==0.2.38 langchain-community==0.2.16 langchain-groq

Found existing installation: langchain 1.2.17
Uninstalling langchain-1.2.17:
  Successfully uninstalled langchain-1.2.17
Found existing installation: langchain-core 1.3.3
Uninstalling langchain-core-1.3.3:
  Successfully uninstalled langchain-core-1.3.3
Found existing installation: langchain-community 0.4.1
Uninstalling langchain-community-0.4.1:
  Successfully uninstalled langchain-community-0.4.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-groq to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-groq to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to 

## 🧪 Step 5 — Test the RAG Chain (without UI)

In [5]:
from langchain_groq import ChatGroq
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory

# Initialize LLM
llm = ChatGroq(
    model='llama-3.1-8b-instant',  # Fast and free on Groq
    temperature=0.3,
    max_tokens=1024,
    groq_api_key=GROQ_API_KEY
)

# Memory: keeps last 5 exchanges
memory = ConversationBufferWindowMemory(
    k=5,
    memory_key='chat_history',
    return_messages=True,
    output_key='answer'
)

# Retriever: fetch top-3 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type='mmr',  # Maximal Marginal Relevance for diversity
    search_kwargs={'k': 3, 'fetch_k': 6}
)

# Full RAG Chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False
)

# --- Test it ---
def ask(question):
    result = rag_chain.invoke({'question': question})
    print(f'\n❓ Q: {question}')
    print(f'\n🤖 A: {result["answer"]}')
    print(f'\n📎 Sources: {[d.metadata["title"] for d in result["source_documents"]]}')
    print('-'*60)

ask('What is a large language model?')
ask('How does it relate to what you just told me?')  # Tests memory!


❓ Q: What is a large language model?

🤖 A: A large language model (LLM) is a type of neural network trained on a vast amount of text data for natural language processing tasks, especially language generation. LLMs can perform various tasks such as generating, summarizing, translating, and parsing text in many contexts. They are a foundational technology behind modern chatbots.

📎 Sources: ['Large language model', 'Large language model', 'Large language model']
------------------------------------------------------------

❓ Q: How does it relate to what you just told me?

🤖 A: I didn't tell you anything prior to your question, so I'm assuming you're asking how the context I provided relates to the general topic of artificial intelligence (AI) and language models like myself.

The context I provided explains how AI models, including language models, can generate false or misleading information due to a lack of understanding of context. This is particularly relevant to deep learning mode

In [7]:
#saving vectorscore for deploying
vectorstore.save_local("vectorstore")

In [8]:
import shutil
from google.colab import files

shutil.make_archive("vectorstore", 'zip', "vectorstore")

files.download("vectorstore.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>